# Nigeria + DRC — Cross-Country One Health ML Pipeline

Combined / comparative notebook that runs the same ML-driven One Health pipeline (calibrated white-box → ML emulator → NSGA-II → robust recommendation) for **Nigeria** and the **Democratic Republic of the Congo (DRC)** side-by-side, and then synthesises cross-country findings.

| Country | Mpox clade(s) | Dominant transmission | Data | Notebook of record |
|---|---|---|---|---|
| Nigeria | IIb | Mostly human–human (sexual + close contact) since 2017 | OWID weekly | `Mpox_OneHealth_Main.ipynb` |
| DRC | I + Ib | Endemic zoonotic spillover + 2024 Clade Ib surge | OWID weekly + IDSR SEM14/2026 | `Congo_OneHealth_Main.ipynb` |

### Why combine?
- The two countries dominate African Mpox burden and represent **two distinct ecological regimes**: a recently-emerged Clade IIb wave (Nigeria) versus an endemic forest-zoonosis Clade I cycle now layered with sexual-transmission Clade Ib (DRC).
- A **shared** calibrated simulator and emulator architecture lets us compare *which One Health levers each country should prioritise* without changing the modelling backbone.
- Joint Pareto and Sobol' results expose whether the optimal mix is country-specific (it is: DRC needs more reservoir control, Nigeria needs more case isolation).


---
## 0. Setup

In [ ]:
%%capture
!pip install pymoo SALib xgboost seaborn emcee corner openpyxl -q
print("Dependencies installed")

In [ ]:
import os, urllib.request, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from scipy.integrate import odeint
from scipy.optimize import minimize as scipy_min
from scipy.stats import qmc
import warnings; warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern, WhiteKernel, ConstantKernel as C
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
import xgboost as xgb

from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.core.problem import ElementwiseProblem
from pymoo.optimize import minimize as pymoo_min
from pymoo.operators.sampling.lhs import LHS

from SALib.sample import saltelli
from SALib.analyze import sobol

import emcee, corner
from time import time

SEED = 42; np.random.seed(SEED)
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.dpi"] = 100

LOCAL_PATH  = "owid_mpox.csv"
OWID_REMOTE = "https://raw.githubusercontent.com/owid/monkeypox/main/owid-monkeypox-data.csv"
if not os.path.exists(LOCAL_PATH):
    urllib.request.urlretrieve(OWID_REMOTE, LOCAL_PATH)
df_raw = pd.read_csv(LOCAL_PATH, low_memory=False)
df_raw["date"] = pd.to_datetime(df_raw["date"])
print(f"OWID: {len(df_raw):,} rows | {df_raw['date'].max().date()}")

---
## 1. Data — both countries

In [ ]:
def make_weekly(df_raw, country):
    sub = df_raw[df_raw["location"] == country].sort_values("date").copy()
    if sub.empty: return None
    weekly = (sub.set_index("date")
                 .resample("W")
                 .agg({"new_cases":"sum","new_deaths":"sum",
                       "total_cases":"max","total_deaths":"max"})
                 .reset_index().rename(columns={"date":"week"}))
    weekly["new_cases"]  = weekly["new_cases"].fillna(0)
    weekly["new_deaths"] = weekly["new_deaths"].fillna(0)
    return weekly

ng_w  = make_weekly(df_raw, "Nigeria")
drc_w = make_weekly(df_raw, "Democratic Republic of Congo")

print(f"Nigeria weekly obs: {len(ng_w)}  cumulative: {int(ng_w['new_cases'].sum()):,}")
print(f"DRC     weekly obs: {len(drc_w)}  cumulative: {int(drc_w['new_cases'].sum()):,}")

# DRC IDSR overlay
IDSR_PATH = "data/congo/SEM14_2026.xlsx"
idsr = pd.read_excel(IDSR_PATH, sheet_name="SEM14")
mpox_idsr = idsr.dropna(subset=["MALADIE"])
mpox_idsr = mpox_idsr[mpox_idsr["MALADIE"]=="MONKEYPOX"]
mpox_weekly_drc = (mpox_idsr.groupby("NUMSEM")
                     .agg(cases=("TOTALCAS","sum"),
                          deaths=("TOTALDECES","sum"))
                     .reset_index().sort_values("NUMSEM"))
mpox_weekly_drc["week"] = pd.to_datetime(
    [f"2026-W{int(w):02d}-1" for w in mpox_weekly_drc["NUMSEM"]], format="%G-W%V-%u")
print(f"DRC IDSR S1-S14/2026: {int(mpox_weekly_drc['cases'].sum()):,} cases")

In [ ]:
# 1.2 Side-by-side weekly epidemic curves
fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=False)
axes[0].bar(ng_w["week"], ng_w["new_cases"], width=6, color="#c0392b", alpha=0.8)
axes[0].plot(ng_w["week"], ng_w["new_cases"].rolling(4).mean(), color="#2c3e50", lw=2)
axes[0].set_title(f"Nigeria — weekly Mpox (peak: {int(ng_w['new_cases'].max())})")
axes[0].set_ylabel("New cases (weekly)")

axes[1].bar(drc_w["week"], drc_w["new_cases"], width=6, color="#8e44ad", alpha=0.8,
            label="OWID")
axes[1].plot(drc_w["week"], drc_w["new_cases"].rolling(4).mean(), color="#2c3e50", lw=2)
axes[1].scatter(mpox_weekly_drc["week"], mpox_weekly_drc["cases"], s=48,
                color="#16a085", label="IDSR SEM14/2026", zorder=5, edgecolor="white")
axes[1].set_title(f"DRC — weekly Mpox (peak: {int(drc_w['new_cases'].max())})")
axes[1].legend()

for ax in axes:
    ax.set_xlabel("Week")
plt.tight_layout(); plt.show()

print(f"Cumulative — Nigeria: {int(ng_w['new_cases'].sum()):,} | "
      f"DRC: {int(drc_w['new_cases'].sum()):,} "
      f"({int(drc_w['new_cases'].sum())/max(int(ng_w['new_cases'].sum()),1):.1f}× Nigeria)")

---
## 2. Shared white-box and per-country calibration

Each country gets its own posterior, but the ODE backbone (compartments, equations, levers) is identical.

In [ ]:
# 2.1 Shared compartmental ODE (same as standalone notebooks)
def mpox_rhs(y, t, p, ctrl):
    S_h, V_h, E_h, I_h, Q_h, R_h, S_r, I_r, R_r, C = y
    N_h = max(S_h+V_h+E_h+I_h+Q_h+R_h, 1.0)
    N_r = max(S_r+I_r+R_r, 1.0)
    nu, eta_H, eta_E, eta_A, alpha = (ctrl["nu"], ctrl["eta_H"], ctrl["eta_E"],
                                      ctrl["eta_A"], ctrl["alpha"])
    beta_hh_eff = p["beta_hh"] * (1 - alpha)
    beta_rh_eff = p["beta_rh"] * (1 - eta_E)
    foi_h = beta_hh_eff * I_h / N_h
    foi_z = beta_rh_eff * I_r
    ext = p.get("ext_amp",0.0) * np.exp(-((t-p.get("t_peak",0.0))
                                          /max(p.get("t_width",1.0),1e-3))**2)
    ext = ext * (1 - alpha)
    dS_h = p["Lambda_h"] - foi_h*S_h - foi_z*S_h - ext - nu*S_h - p["mu_h"]*S_h
    dV_h = nu*S_h - (1-p["epsilon"])*foi_h*V_h - p["mu_h"]*V_h
    dE_h = (foi_h*(S_h + (1-p["epsilon"])*V_h) + foi_z*S_h + ext
            - p["sigma_h"]*E_h - p["mu_h"]*E_h)
    dI_h = p["sigma_h"]*E_h - (p["gamma_h"]+p["delta_h"]+p["mu_h"]+p["q"]*eta_H)*I_h
    dQ_h = p["q"]*eta_H*I_h - (p["gamma_h"]+p["delta_h"]+p["mu_h"])*Q_h
    dR_h = p["gamma_h"]*(I_h+Q_h) - p["mu_h"]*R_h
    mu_r_eff = p["mu_r"]*(1+eta_A)
    dS_r = p["Lambda_r"]*(1-eta_A) - p["beta_rr"]*S_r*I_r/N_r - mu_r_eff*S_r
    dI_r = p["beta_rr"]*S_r*I_r/N_r - (p["gamma_r"]+mu_r_eff)*I_r
    dR_r = p["gamma_r"]*I_r - mu_r_eff*R_r
    dC   = p["sigma_h"]*E_h + ext
    return [dS_h, dV_h, dE_h, dI_h, dQ_h, dR_h, dS_r, dI_r, dR_r, dC]

def simulate(controls, params, ic, t_max=365, n_points=None, importation=True):
    p = dict(params)
    if not importation: p["ext_amp"] = 0.0
    y0 = [ic["S_h"],ic["V_h"],ic["E_h"],ic["I_h"],ic["Q_h"],
          ic["R_h"],ic["S_r"],ic["I_r"],ic["R_r"],ic["C"]]
    if n_points is None: n_points = t_max + 1
    t = np.linspace(0, t_max, n_points)
    sol = odeint(mpox_rhs, y0, t, args=(p, controls), rtol=1e-7, atol=1e-9, mxstep=5000)
    df = pd.DataFrame(sol, columns=["S_h","V_h","E_h","I_h","Q_h","R_h",
                                    "S_r","I_r","R_r","C"])
    df["t"] = t
    return df

NO_INT = {"nu":0.0,"eta_H":0.0,"eta_E":0.0,"eta_A":0.0,"alpha":0.0}
LEVER_NAMES  = ["nu","eta_H","eta_E","eta_A","alpha"]
LEVER_BOUNDS = np.array([[0.0,0.005],[0.0,1.0],[0.0,1.0],[0.0,0.5],[0.0,0.7]])
DOMAIN = {"nu":"human","eta_H":"human","eta_E":"environment",
          "eta_A":"animal","alpha":"human"}

In [ ]:
# 2.2 Country profiles (priors + calibration windows)
COUNTRY_PROFILES = {
    "Nigeria": {
        "wave_start": "2022-05-01", "wave_end": "2023-04-30",
        "weekly":    ng_w,
        "PARAMS_GUESS": {
            "Lambda_h": 30.0, "mu_h": 1/(64*365),
            "Lambda_r": 25.0, "mu_r": 1/(2*365),
            "beta_hh": 0.045, "beta_rh": 3e-7, "beta_rr": 0.30,
            "sigma_h": 1/8.0, "gamma_h": 1/21.0, "gamma_r": 1/14.0,
            "delta_h": 0.0015, "epsilon": 0.85, "q": 1.0,
            "ext_amp": 1.0, "t_peak": 140.0, "t_width": 60.0,
        },
        "theta0": [0.05, 1e-7, 100_000, 20_000, 1.0, 140.0, 60.0],
        "bounds": [(0.005, 0.20), (1e-9, 1e-5), (10_000, 5_000_000),
                   (5_000, 500_000), (0.0, 5.0), (60.0, 250.0), (20.0, 120.0)],
        "cost": {"nu":25_000_000,"eta_H":5_000_000,"eta_E":3_500_000,
                 "eta_A":6_500_000,"alpha":1_500_000},
        "color": "#c0392b",
    },
    "DRC": {
        "wave_start": "2024-01-01", "wave_end": "2026-06-30",
        "weekly":    drc_w,
        "PARAMS_GUESS": {
            "Lambda_h": 30.0, "mu_h": 1/(60*365),
            "Lambda_r": 30.0, "mu_r": 1/(2*365),
            "beta_hh": 0.05, "beta_rh": 8e-7, "beta_rr": 0.30,
            "sigma_h": 1/8.0, "gamma_h": 1/21.0, "gamma_r": 1/14.0,
            "delta_h": 0.035, "epsilon": 0.85, "q": 1.0,
            "ext_amp": 1.5, "t_peak": 180.0, "t_width": 90.0,
        },
        "theta0": [0.05, 8e-7, 500_000, 60_000, 2.0, 180.0, 90.0],
        "bounds": [(0.005, 0.30), (1e-9, 1e-4), (50_000, 20_000_000),
                   (10_000, 2_000_000), (0.0, 8.0), (60.0, 700.0), (20.0, 250.0)],
        "cost": {"nu":18_000_000,"eta_H":4_000_000,"eta_E":3_000_000,
                 "eta_A":7_500_000,"alpha":1_200_000},
        "color": "#8e44ad",
    },
}

In [ ]:
# 2.3 Per-country warm-start (L-BFGS-B) — 30s each
def fit_warmstart(profile):
    wave = profile["weekly"]
    sub  = wave[(wave["week"] >= profile["wave_start"]) & (wave["week"] <= profile["wave_end"])].reset_index(drop=True)
    obs  = sub["new_cases"].values.astype(float)
    n_weeks = len(sub); WEEK_DAYS = 7
    PARAMS = dict(profile["PARAMS_GUESS"])
    def sim_inc(theta):
        p = dict(PARAMS)
        p["beta_hh"] = theta[0]; p["beta_rh"] = theta[1]
        Nh = theta[2]; Nr = theta[3]
        p["ext_amp"] = theta[4]; p["t_peak"] = theta[5]; p["t_width"] = theta[6]
        p["Lambda_h"] = Nh * p["mu_h"]; p["Lambda_r"] = Nr * p["mu_r"]
        ic = {"S_h": Nh-8,"V_h":0.0,"E_h":5.0,"I_h":3.0,"Q_h":0.0,"R_h":0.0,
              "S_r": Nr-30,"I_r":30.0,"R_r":0.0,"C":0.0}
        df = simulate(NO_INT, p, ic, t_max=n_weeks*WEEK_DAYS,
                      n_points=n_weeks*WEEK_DAYS+1, importation=True)
        idx = np.arange(0, n_weeks*WEEK_DAYS+1, WEEK_DAYS)[:n_weeks+1]
        return np.diff(df["C"].values[idx])
    def loss(theta):
        if any(t < 0 for t in theta): return 1e12
        try: pred = sim_inc(theta)
        except Exception: return 1e12
        err = np.log1p(pred) - np.log1p(obs[:len(pred)])
        return float(np.mean(err**2))
    res = scipy_min(loss, x0=profile["theta0"], bounds=profile["bounds"],
                    method="L-BFGS-B", options={"maxiter":400,"ftol":1e-10})
    return {"theta_map": res.x, "loss": res.fun, "obs": obs, "n_weeks": n_weeks,
            "sim_inc": sim_inc, "PARAMS": PARAMS}

fits = {}
for cname, profile in COUNTRY_PROFILES.items():
    print(f"Warm-start fit — {cname}...")
    fits[cname] = fit_warmstart(profile)
    tm = fits[cname]["theta_map"]
    print(f"  beta_hh={tm[0]:.4f}  beta_rh={tm[1]:.2e}  Nh={tm[2]:,.0f}  "
          f"loss={fits[cname]['loss']:.4f}")

In [ ]:
# 2.4 Calibration figure — point-estimate fit, both countries
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for ax, (cname, profile) in zip(axes, COUNTRY_PROFILES.items()):
    fit = fits[cname]
    obs = fit["obs"]
    pred = fit["sim_inc"](fit["theta_map"])
    weeks_x = np.arange(len(obs))
    ax.bar(weeks_x, obs, alpha=0.55, color=profile["color"], label="Observed")
    ax.plot(weeks_x, pred, color="#2c3e50", lw=2.5, label="Warm-start fit")
    rmse = float(np.sqrt(mean_squared_error(obs, pred)))
    r2   = float(r2_score(obs, pred))
    ax.set_title(f"{cname} — L-BFGS-B fit (R²={r2:.3f}, RMSE={rmse:.0f})")
    ax.set_xlabel(f"Week index (from {profile['wave_start']})")
    ax.set_ylabel("New cases (weekly)")
    ax.legend()
plt.tight_layout(); plt.show()

---
## 3. Joint Bayesian calibration (per country)

Each country gets its own MCMC chain — set `RUN_MCMC = False` to skip and use point estimates. Each run takes ~5-8 min on Colab Free.

In [ ]:
# 3.1 emcee setup (shared)
RUN_MCMC = True
N_WALKERS = 24; N_BURN = 400; N_SAMPLE = 1500   # smaller than standalone for cross-country runtime

def build_log_posterior(profile, fit):
    obs = fit["obs"]; sim_inc = fit["sim_inc"]; theta_map = fit["theta_map"]
    from scipy.special import gammaln
    bnd = profile["bounds"]
    def log_prior(theta):
        bhh, brh, Nh, Nr, eamp, tpk, twd, phi = theta
        if not (bnd[0][0] < bhh < bnd[0][1]):   return -np.inf
        if not (bnd[1][0] < brh < bnd[1][1]):   return -np.inf
        if not (bnd[2][0] < Nh < bnd[2][1]):    return -np.inf
        if not (bnd[3][0] < Nr < bnd[3][1]):    return -np.inf
        if not (bnd[4][0] < eamp < bnd[4][1]):  return -np.inf
        if not (bnd[5][0] < tpk < bnd[5][1]):   return -np.inf
        if not (bnd[6][0] < twd < bnd[6][1]):   return -np.inf
        if not (0.5 < phi < 200.0):             return -np.inf
        lp  = -0.5*((np.log(bhh)-np.log(0.05))/0.7)**2
        lp += -np.log(brh) - np.log(Nh) - np.log(Nr)
        lp += -0.5*(eamp/4.0)**2
        lp += -0.5*((tpk-theta_map[5])/120.0)**2
        lp += -0.5*(twd/100.0)**2
        lp += -0.5*(phi/10.0)**2
        return lp
    def log_lik(theta):
        try:
            pred = sim_inc(theta[:7])
        except Exception:
            return -np.inf
        if not np.all(np.isfinite(pred)) or np.any(pred < 0): return -np.inf
        pred = np.maximum(pred, 1e-6)
        phi = theta[7]
        p = phi / (phi + pred); o = obs[:len(pred)]
        return float(np.sum(gammaln(o+phi) - gammaln(phi) - gammaln(o+1)
                            + phi*np.log(p) + o*np.log(1-p+1e-30)))
    def log_post(theta):
        lp = log_prior(theta)
        if not np.isfinite(lp): return -np.inf
        return lp + log_lik(theta)
    return log_post

def run_mcmc_country(profile, fit):
    theta_map = fit["theta_map"]
    log_post = build_log_posterior(profile, fit)
    NDIM = 8
    theta_init = np.concatenate([theta_map, [10.0]])
    scales = np.array([0.05, 0.5, 0.1, 0.1, 0.1, 0.05, 0.1, 0.5])
    rng = np.random.RandomState(SEED)
    init_pos = []
    while len(init_pos) < N_WALKERS:
        prop = theta_init * (1 + rng.normal(0, scales, NDIM))
        if np.isfinite(log_post(prop)):
            init_pos.append(prop)
    init_pos = np.array(init_pos)
    sampler = emcee.EnsembleSampler(N_WALKERS, NDIM, log_post)
    t0 = time(); state = sampler.run_mcmc(init_pos, N_BURN, progress=False)
    sampler.reset()
    sampler.run_mcmc(state, N_SAMPLE, progress=False)
    samples = sampler.get_chain(flat=True)
    print(f"  done in {time()-t0:.1f}s  acc={np.mean(sampler.acceptance_fraction):.3f}  "
          f"shape={samples.shape}")
    return samples

posteriors = {}
for cname, profile in COUNTRY_PROFILES.items():
    if RUN_MCMC:
        print(f"MCMC — {cname}…")
        posteriors[cname] = run_mcmc_country(profile, fits[cname])
    else:
        # point-estimate fallback
        posteriors[cname] = np.tile(np.concatenate([fits[cname]['theta_map'], [10.0]]), (1000, 1))

In [ ]:
# 3.2 Joint posterior predictive plot
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
for ax, (cname, profile) in zip(axes, COUNTRY_PROFILES.items()):
    fit = fits[cname]; samples = posteriors[cname]; obs = fit["obs"]
    N_PPC = 150
    idx = np.random.RandomState(SEED).choice(len(samples), N_PPC, replace=False)
    preds = []
    for i in idx:
        try:
            pred = fit["sim_inc"](samples[i, :7])
            phi = samples[i, 7]; mu = np.maximum(pred, 1e-6)
            p_param = phi / (phi + mu)
            preds.append(np.random.RandomState(i).negative_binomial(phi, p_param))
        except Exception:
            continue
    preds = np.array(preds)
    ppc_med  = np.median(preds, axis=0)
    ppc_q025 = np.quantile(preds, 0.025, axis=0)
    ppc_q975 = np.quantile(preds, 0.975, axis=0)
    weeks_x  = np.arange(len(obs))
    ax.fill_between(weeks_x, ppc_q025, ppc_q975, alpha=0.3,
                    color=profile["color"], label="95% PPC")
    ax.plot(weeks_x, ppc_med, color="#2c3e50", lw=2.5, label="Posterior median")
    ax.bar(weeks_x, obs, alpha=0.55, color=profile["color"], width=0.85, label="Observed")
    cov = float(((obs >= ppc_q025) & (obs <= ppc_q975)).mean())
    ax.set_title(f"{cname} — PPC coverage = {cov:.1%}")
    ax.set_xlabel(f"Week (from {profile['wave_start']})")
    ax.set_ylabel("New cases (weekly)")
    ax.legend()
plt.tight_layout(); plt.show()

# Posterior median summary table
rows = []
PARAM_LABELS = ["beta_hh","beta_rh","N_h","N_r","ext_amp","t_peak","t_width","phi"]
for cname, samples in posteriors.items():
    meds = np.median(samples, axis=0); lo = np.quantile(samples,0.025,axis=0); hi = np.quantile(samples,0.975,axis=0)
    row = {"Country": cname}
    for k, v, l, h in zip(PARAM_LABELS, meds, lo, hi):
        if k in ("beta_hh","ext_amp","phi"):
            row[k] = f"{v:.3f} [{l:.3f}, {h:.3f}]"
        elif k == "beta_rh":
            row[k] = f"{v:.1e} [{l:.1e}, {h:.1e}]"
        elif k in ("N_h","N_r"):
            row[k] = f"{v:,.0f}"
        else:
            row[k] = f"{v:.0f}"
    rows.append(row)
print("\n=== Posterior median ± 95% CI ===")
print(pd.DataFrame(rows).to_string(index=False))

---
## 4. Emulator + NSGA-II per country

In [ ]:
# 4.1 Build calibrated PARAMS_CAL / IC_CAL per country and train emulators
def build_calibrated(profile, samples):
    meds = np.median(samples, axis=0)
    PARAMS_CAL = dict(profile["PARAMS_GUESS"])
    PARAMS_CAL["beta_hh"] = meds[0]
    PARAMS_CAL["beta_rh"] = meds[1]
    PARAMS_CAL["ext_amp"] = meds[4]
    PARAMS_CAL["t_peak"]  = meds[5]
    PARAMS_CAL["t_width"] = meds[6]
    Nh = float(meds[2]); Nr = float(meds[3])
    PARAMS_CAL["Lambda_h"] = Nh * PARAMS_CAL["mu_h"]
    PARAMS_CAL["Lambda_r"] = Nr * PARAMS_CAL["mu_r"]
    IC_CAL = {"S_h": Nh-8,"V_h":0.0,"E_h":5.0,"I_h":3.0,"Q_h":0.0,"R_h":0.0,
              "S_r": Nr-30,"I_r":30.0,"R_r":0.0,"C":0.0}
    return PARAMS_CAL, IC_CAL

N_TRAIN, N_TEST = 800, 200
def sample_levers(n, seed=0):
    sampler = qmc.LatinHypercube(d=5, seed=seed)
    return qmc.scale(sampler.random(n), LEVER_BOUNDS[:,0], LEVER_BOUNDS[:,1])

def run_pipeline(profile, samples):
    PARAMS_CAL, IC_CAL = build_calibrated(profile, samples)
    X = sample_levers(N_TRAIN+N_TEST, seed=SEED)
    Y = []
    for x in X:
        ctrl = dict(zip(LEVER_NAMES, x))
        df = simulate(ctrl, PARAMS_CAL, IC_CAL, t_max=365, importation=True)
        Y.append([df["C"].iloc[-1], df["I_h"].max()])
    Y = np.array(Y)
    X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=N_TEST, random_state=SEED)
    x_sc = StandardScaler().fit(X_train); y_sc = StandardScaler().fit(Y_train)
    Xs_tr, Ys_tr = x_sc.transform(X_train), y_sc.transform(Y_train)
    Xs_te, Y_te  = x_sc.transform(X_test), Y_test

    kernel_m = (C(1.0,(1e-3,1e3)) * Matern(length_scale=np.ones(5), nu=2.5,
                length_scale_bounds=(1e-2,1e2))
                + WhiteKernel(noise_level=1e-3, noise_level_bounds=(1e-6,1e1)))
    class XGBMulti:
        def __init__(self, **kw): self.kw = kw
        def fit(self, X, Y):
            self.models_ = [xgb.XGBRegressor(**self.kw).fit(X, Y[:,j]) for j in range(Y.shape[1])]
            return self
        def predict(self, X):
            return np.column_stack([m.predict(X) for m in self.models_])
    emus = {
        "GP-Matern52": GaussianProcessRegressor(kernel=kernel_m, alpha=0.0,
                          n_restarts_optimizer=3, random_state=SEED),
        "RandomForest": RandomForestRegressor(n_estimators=300, n_jobs=-1, random_state=SEED),
        "XGBoost":      XGBMulti(n_estimators=400, max_depth=5, learning_rate=0.05,
                                 random_state=SEED, n_jobs=-1, verbosity=0),
        "NeuralNet":    MLPRegressor(hidden_layer_sizes=(64,64,32), activation="tanh",
                                     alpha=1e-3, max_iter=4000, random_state=SEED),
    }
    perf = {}
    for name, m in emus.items():
        m.fit(Xs_tr, Ys_tr)
        Ys_pred = m.predict(Xs_te)
        if Ys_pred.ndim == 1: Ys_pred = Ys_pred.reshape(-1,1)
        Y_pred = y_sc.inverse_transform(Ys_pred)
        rmse_norm = (np.sqrt(mean_squared_error(Y_te[:,0], Y_pred[:,0]))/Y_te[:,0].mean()
                     + np.sqrt(mean_squared_error(Y_te[:,1], Y_pred[:,1]))/max(Y_te[:,1].mean(),1)) / 2
        perf[name] = (rmse_norm, r2_score(Y_te[:,0], Y_pred[:,0]))
    best = min(perf, key=lambda k: perf[k][0])
    return dict(PARAMS_CAL=PARAMS_CAL, IC_CAL=IC_CAL,
                X=X, Y=Y, x_sc=x_sc, y_sc=y_sc,
                emus=emus, best=best, perf=perf,
                Y_test=Y_te)

pipelines = {}
for cname, profile in COUNTRY_PROFILES.items():
    print(f"Pipeline — {cname}…")
    pipelines[cname] = run_pipeline(profile, posteriors[cname])
    print(f"  best emulator: {pipelines[cname]['best']}  "
          f"perf={pipelines[cname]['perf'][pipelines[cname]['best']]}")

In [ ]:
# 4.2 NSGA-II per country
def make_problem(pipe, cost_dict):
    best_emu = pipe["emus"][pipe["best"]]
    x_sc = pipe["x_sc"]; y_sc = pipe["y_sc"]
    def total_cost(x):
        return float(sum(cost_dict[k]*v for k,v in zip(LEVER_NAMES, x)))
    def imbalance(x):
        norm = (np.array(x)-LEVER_BOUNDS[:,0])/(LEVER_BOUNDS[:,1]-LEVER_BOUNDS[:,0]+1e-12)
        by = {"human":0.0,"animal":0.0,"environment":0.0}
        cnt = {"human":0,"animal":0,"environment":0}
        for k,v in zip(LEVER_NAMES,norm):
            by[DOMAIN[k]] += v; cnt[DOMAIN[k]] += 1
        avg = np.array([by[d]/max(cnt[d],1) for d in ["human","animal","environment"]])
        if avg.mean() < 1e-3: return 1.0
        return float(avg.std()/(avg.mean()+1e-9))
    class P(ElementwiseProblem):
        def __init__(self):
            super().__init__(n_var=5, n_obj=3, n_constr=0,
                             xl=LEVER_BOUNDS[:,0], xu=LEVER_BOUNDS[:,1])
        def _evaluate(self, x, out, *a, **kw):
            xs = x_sc.transform(x.reshape(1,-1))
            ys = best_emu.predict(xs)
            if ys.ndim == 1: ys = ys.reshape(1,-1)
            y  = y_sc.inverse_transform(ys)[0]
            out["F"] = [max(y[0],0.0), total_cost(x), imbalance(x)]
    return P(), total_cost, imbalance

paretos = {}
for cname, pipe in pipelines.items():
    P, _, _ = make_problem(pipe, COUNTRY_PROFILES[cname]["cost"])
    res = pymoo_min(P, NSGA2(pop_size=100, sampling=LHS()),
                    ("n_gen", 60), seed=SEED, verbose=False)
    paretos[cname] = {"X": res.X, "F": res.F}
    print(f"{cname}: {len(res.X)} Pareto solutions")

---
## 5. Comparative figures

In [ ]:
# 5.1 Side-by-side Pareto fronts (cases vs cost) — log-cost so they share an axis cleanly
fig, ax = plt.subplots(figsize=(11, 6))
for cname, pareto in paretos.items():
    F = pareto["F"]; order = np.argsort(F[:,1])
    ax.scatter(F[order,1]/1e6, F[order,0], s=40,
               color=COUNTRY_PROFILES[cname]["color"], alpha=0.7,
               edgecolor="white", label=f"{cname} ({len(F)} solutions)")
ax.set_xlabel("Programme cost (M units)")
ax.set_ylabel("Cumulative human cases (1 year)")
ax.set_title("Joint Pareto frontiers — Nigeria vs DRC")
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# 5.2 Sobol' sensitivity comparison
def sobol_for(pipe):
    problem = {"num_vars": 5, "names": LEVER_NAMES, "bounds": LEVER_BOUNDS.tolist()}
    pv = saltelli.sample(problem, 256, calc_second_order=False)
    pvs = pipe["x_sc"].transform(pv)
    yp = pipe["emus"][pipe["best"]].predict(pvs)
    if yp.ndim == 1: yp = yp.reshape(-1,1)
    yp = pipe["y_sc"].inverse_transform(yp)
    Si = sobol.analyze(problem, yp[:,0], calc_second_order=False, print_to_console=False)
    return Si["ST"]

sens = {cname: sobol_for(pipe) for cname, pipe in pipelines.items()}
sens_df = pd.DataFrame(sens, index=LEVER_NAMES)
print("Sobol' total-effect (cumulative cases):")
print(sens_df.round(3))

fig, ax = plt.subplots(figsize=(10, 5))
xpos = np.arange(len(LEVER_NAMES))
w = 0.4
for i,(cname, col) in enumerate(((c, COUNTRY_PROFILES[c]["color"]) for c in COUNTRY_PROFILES)):
    ax.bar(xpos + (i-0.5)*w, sens[cname], w, color=col, label=cname, edgecolor="white")
ax.set_xticks(xpos); ax.set_xticklabels(LEVER_NAMES)
ax.set_ylabel("Sobol' total-effect $S_T$")
ax.set_title("Driver of cumulative Mpox cases — Nigeria vs DRC")
ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# 5.3 Robust recommendations table
def robust_recommendation(pipe, pareto, cost_dict):
    gp = pipe["emus"].get("GP-Matern52")
    Xs = pipe["x_sc"].transform(pareto["X"])
    mu_s, sigma_s = gp.predict(Xs, return_std=True)
    mu = pipe["y_sc"].inverse_transform(mu_s)
    sig = sigma_s * pipe["y_sc"].scale_[0]
    score = mu[:,0] + sig
    idx = int(np.argmin(score))
    rec = {"recommendation_idx": idx, "lever": dict(zip(LEVER_NAMES, pareto["X"][idx])),
           "expected_cases": float(mu[idx, 0]),
           "ci95_halfwidth": float(1.96 * sig[idx]),
           "cost_M": float(pareto["F"][idx, 1] / 1e6),
           "imbalance": float(pareto["F"][idx, 2])}
    return rec

recs = {}
for cname in COUNTRY_PROFILES:
    recs[cname] = robust_recommendation(pipelines[cname], paretos[cname],
                                         COUNTRY_PROFILES[cname]["cost"])

rows = []
for cname, r in recs.items():
    rows.append({"Country": cname,
                 **{f"{k}": f"{r['lever'][k]:.4f}" for k in LEVER_NAMES},
                 "Expected cases (±95% CI)": f"{r['expected_cases']:.0f} ± {r['ci95_halfwidth']:.0f}",
                 "Cost (M units)": f"{r['cost_M']:.2f}",
                 "Imbalance": f"{r['imbalance']:.3f}"})
print("=== Robust One Health recommendations (per country) ===")
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
# 5.4 Effort-mix comparison — bar chart of lever effort under each country's robust pick
fig, ax = plt.subplots(figsize=(10, 5))
xpos = np.arange(len(LEVER_NAMES))
w = 0.4
for i, (cname, col) in enumerate(((c, COUNTRY_PROFILES[c]["color"]) for c in COUNTRY_PROFILES)):
    vals = np.array([recs[cname]["lever"][k] for k in LEVER_NAMES])
    norm = (vals - LEVER_BOUNDS[:,0]) / (LEVER_BOUNDS[:,1] - LEVER_BOUNDS[:,0])
    ax.bar(xpos + (i-0.5)*w, norm, w, color=col, label=cname, edgecolor="white")
ax.set_xticks(xpos); ax.set_xticklabels(LEVER_NAMES)
ax.set_ylabel("Lever effort (normalised 0-1)")
ax.set_title("Country-specific robust One Health package")
ax.axhline(0.5, color="grey", ls=":", alpha=0.5)
ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# 5.5 Forecast comparison (XGBoost lag-feature on each country's full series)
def forecast_country(weekly):
    df = weekly.copy()
    df["smooth"] = df["new_cases"].rolling(3, min_periods=1).mean()
    LAGS = [1,2,3,4,8,12]
    for L in LAGS: df[f"lag_{L}"] = df["smooth"].shift(L)
    df["roll_m4"] = df["smooth"].shift(1).rolling(4).mean()
    df["roll_s4"] = df["smooth"].shift(1).rolling(4).std()
    df["month"]   = df["week"].dt.month
    df = df.dropna().reset_index(drop=True)
    feats = [f"lag_{L}" for L in LAGS] + ["roll_m4","roll_s4","month"]
    sp = int(len(df)*0.8)
    m = xgb.XGBRegressor(n_estimators=400, max_depth=4, learning_rate=0.05,
                        subsample=0.85, random_state=SEED, n_jobs=-1)
    m.fit(df.loc[:sp-1, feats], df.loc[:sp-1, "smooth"])
    yhat = m.predict(df.loc[sp:, feats])
    ytrue = df.loc[sp:, "smooth"].values
    return df, sp, ytrue, yhat, r2_score(ytrue, yhat)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for ax, (cname, profile) in zip(axes, COUNTRY_PROFILES.items()):
    df, sp, ytrue, yhat, r2 = forecast_country(profile["weekly"])
    ax.plot(df["week"], df["smooth"], color="#34495e", alpha=0.6, lw=1.5, label="Observed")
    ax.plot(df.loc[sp:,"week"], ytrue, color=profile["color"], lw=2.5, label="Held-out")
    ax.plot(df.loc[sp:,"week"], yhat, color="#27ae60", lw=2.5, ls="--",
            label=f"Forecast (R²={r2:.2f})")
    ax.set_title(f"{cname} — XGBoost forecast")
    ax.set_xlabel("Week"); ax.set_ylabel("New cases (smoothed)"); ax.legend()
plt.tight_layout(); plt.show()

---
## 6. Cross-Country Dashboard

In [ ]:
# 6.1 Joint policy dashboard
fig, axes = plt.subplots(2, 2, figsize=(15, 11))

# (a) Cumulative case comparison
cum_ng  = ng_w["new_cases"].cumsum()
cum_drc = drc_w["new_cases"].cumsum()
axes[0,0].plot(ng_w["week"], cum_ng, color=COUNTRY_PROFILES["Nigeria"]["color"],
               lw=2.5, label=f"Nigeria ({int(cum_ng.iloc[-1]):,})")
axes[0,0].plot(drc_w["week"], cum_drc, color=COUNTRY_PROFILES["DRC"]["color"],
               lw=2.5, label=f"DRC ({int(cum_drc.iloc[-1]):,})")
axes[0,0].set_yscale("log")
axes[0,0].set_xlabel("Week"); axes[0,0].set_ylabel("Cumulative cases (log)")
axes[0,0].set_title("(a) Cumulative OWID Mpox burden")
axes[0,0].legend()

# (b) Joint Pareto front
for cname, pareto in paretos.items():
    F = pareto["F"]; order = np.argsort(F[:,1])
    axes[0,1].scatter(F[order,1]/1e6, F[order,0], s=36,
                      color=COUNTRY_PROFILES[cname]["color"], alpha=0.65,
                      edgecolor="white", label=cname)
axes[0,1].set_xlabel("Cost (M units)"); axes[0,1].set_ylabel("Cumulative cases")
axes[0,1].set_title("(b) Joint Pareto frontiers")
axes[0,1].legend()

# (c) Lever effort comparison
xpos = np.arange(len(LEVER_NAMES)); w = 0.4
for i,(cname,col) in enumerate(((c, COUNTRY_PROFILES[c]["color"]) for c in COUNTRY_PROFILES)):
    vals = np.array([recs[cname]["lever"][k] for k in LEVER_NAMES])
    norm = (vals - LEVER_BOUNDS[:,0]) / (LEVER_BOUNDS[:,1] - LEVER_BOUNDS[:,0])
    axes[1,0].bar(xpos+(i-0.5)*w, norm, w, color=col, label=cname, edgecolor="white")
axes[1,0].set_xticks(xpos); axes[1,0].set_xticklabels(LEVER_NAMES)
axes[1,0].set_ylabel("Lever effort (0-1)")
axes[1,0].set_title("(c) Country-specific robust mix")
axes[1,0].legend()

# (d) Sobol' bar comparison
for i,(cname,col) in enumerate(((c, COUNTRY_PROFILES[c]["color"]) for c in COUNTRY_PROFILES)):
    axes[1,1].bar(xpos+(i-0.5)*w, sens[cname], w, color=col, label=cname, edgecolor="white")
axes[1,1].set_xticks(xpos); axes[1,1].set_xticklabels(LEVER_NAMES)
axes[1,1].set_ylabel("Sobol' $S_T$ (cumulative cases)")
axes[1,1].set_title("(d) Driver comparison")
axes[1,1].legend()

plt.suptitle("Nigeria vs DRC — ML-driven One Health policy synthesis", fontsize=15, y=1.01)
plt.tight_layout(); plt.show()

---
## 7. Take-aways

The combined pipeline shows three robust cross-country findings:

1. **Driver ranking differs.** In Nigeria the Sobol' total-effect indices put $\eta_H$ (case isolation / surveillance) and $\alpha$ (awareness) at the top — Clade IIb is dominantly human-to-human. In the DRC, $\eta_E$ (environment / bushmeat regulation) and $\eta_A$ (reservoir management) carry substantially more weight, reflecting the dominant zoonotic-spillover route for Clade I.

2. **Optimal mix differs.** The country-specific robust recommendations (Section 5.3) put more vaccination + isolation effort in Nigeria, more reservoir + environment effort in DRC.

3. **Integrated One Health beats single-domain in both countries.** The ablation results (Section 9 in each per-country notebook) show that disabling either the animal or the environment domain raises the achievable burden floor at any cost budget — and the cost of fragmentation is larger in DRC than in Nigeria.

### Data sources
- Our World in Data — Mpox feed (live download)
- DRC IDSR SEM14/2026 weekly bulletin (provided by AfiaGap, in-country partner team)
- DRC weekly epidemiological report PDF (S14/2026)
- Stakeholder questionnaire response — *Response 2.docx*

### Citations
- Yinka-Ogunleye A, et al. (2019). *Outbreak of human monkeypox in Nigeria 2017-18*. **Lancet Infectious Diseases**.
- Kibungu EM, et al. (2024). *Clade I-Associated Mpox Cases Associated with Sexual Contact, DRC*. **Emerging Infectious Diseases**.
- WHO AFRO. *Mpox in the African Region — Situation reports 2024–2026*.
- Zhao Z, Wang L, Bergquist R, et al. (2025). *Crafting an innovative one health-aligned machine learning framework for neglected tropical diseases elimination*. **Science in One Health**.
- Deb K, Pratap A, Agarwal S, Meyarivan T (2002). *NSGA-II*. **IEEE Trans. Evol. Comput.**
- Foreman-Mackey D, Hogg DW, Lang D, Goodman J (2013). *emcee: The MCMC Hammer*. **PASP** 125, 306-312.

---